In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings("ignore", category = FutureWarning)

import importlib
import mis_funciones
importlib.reload(mis_funciones)

import gspread
from google.oauth2.service_account import Credentials
from ortools.sat.python import cp_model

In [17]:
# Configuración inicial
id_libro = "1a7vwC8QZAb7KI2elM_vJOZGSQzitnZ0Xw6BWFlG5guw"
id_hoja_operadores = "1408324300"
id_hoja_lista_actividades = "830504512"
id_hoja_areas = "634917592"
id_hoja_WIP = "1742807779"

In [19]:
df_operadores = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                              id_hoja = id_hoja_operadores,
                                              desc_hoja = 'operadores', 
                                              flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 27


,ID_OPERADOR,OPERADOR,AREA,ID_DIA_DESCANSO,DIA_DESCANSO,FECHA_ACTUALIZACION_DESCANSO,FECHA_INGRESO,ESTATUS
0,1,ABIZAID ARAGÓN,ARMADO Y TAPIZADO,1,Lunes,2026-04-19,1900-01-01,ACTIVO
1,2,ANAYELI LÓPEZ,GENERAL,6,Sábado,2026-04-19,1900-01-01,ACTIVO
2,3,AQUILEO GUTIÉRREZ,ESTRUCTURA Y DISEÑO,1,Lunes,2026-04-19,1900-01-01,ACTIVO


💾 Archivo guardado para lectura offline en: offline_databases\informacion_operadores_2026-04-20_23-41-16.csv


In [20]:
df_lista_actividades = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                              id_hoja = id_hoja_lista_actividades,
                                              desc_hoja = 'lista_actividades', 
                                              flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 60


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA
0,1,1,1.1,PROCESAMIENTO DE FIBRA,Preparación de moldes (Limpieza y cera),FIBRAS,1.0,
1,1,2,1.2,PROCESAMIENTO DE FIBRA,Aplicación de Gelcoat,FIBRAS,0.5,1.1
2,1,3,1.3,PROCESAMIENTO DE FIBRA,Laminado de capas de fibra de vidrio,FIBRAS,2.0,1.2


💾 Archivo guardado para lectura offline en: offline_databases\informacion_lista_actividades_2026-04-20_23-41-25.csv


In [21]:
df_areas = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                        id_hoja = id_hoja_areas,
                                        desc_hoja = 'areas', 
                                        flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 15


,ID_AREA,AREA,ID_SUBAREA,SUBAREA,FECHA_CREACION,ESTATUS
0,0,FIBRAS,1,Fibras,1900-01-01,ACTIVO
1,1,ESTRUCTURA Y DISEÑO,1,Diseño,1900-01-01,ACTIVO
2,1,ESTRUCTURA Y DISEÑO,2,Estructura,1900-01-01,ACTIVO


💾 Archivo guardado para lectura offline en: offline_databases\informacion_areas_2026-04-20_23-41-37.csv


In [34]:
df_wip = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                        id_hoja = id_hoja_WIP,
                                        desc_hoja = 'wip', 
                                        flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 10


,ID_UNIDAD_TIPO,TIPO_UNIDAD,ID_UNIDAD,FECHA_INGRESO_PRODUCCION,FECHA_ACTUALIZACION_PRODUCCION,KEY_SUBPROCESO_MAS_RECIENTE,AREA_ACTUAL,PORC_AVANCE_APROX
0,187,CG4,CG4-187,24/3/2026,18/4/2026,9.7,DETALLADO,95%
1,188,CG4,CG4-188,27/3/2026,18/4/2026,9.2,DETALLADO,85%
2,189,CG4,CG4-189,31/3/2026,18/4/2026,8.8,ARMADO Y TAPIZADO,75%


💾 Archivo guardado para lectura offline en: offline_databases\informacion_wip_2026-04-20_23-58-51.csv


In [23]:
def obtener_dia_actual(fecha_simulacion):
    # Traducir nombre del día para el filtro de operadores
    dias = {0: "Lunes", 1: "Mares", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
    return dias[fecha_simulacion.weekday()]

In [24]:
df_lista_actividades['TIEMPO'] = pd.to_numeric(df_lista_actividades['TIEMPO'], errors='coerce').fillna(0)

# Función segura para convertir llaves 1_1 a float 1.1
def key_to_val(k):
    if pd.isna(k) or str(k).strip() == '': 
        return 0.0
    return float(str(k).replace('_', '.'))

In [25]:
fecha_hoy_str = "20/04/2026"
fecha_hoy = datetime.strptime(fecha_hoy_str, "%d/%m/%Y")
dia_nombre = obtener_dia_actual(fecha_hoy)
    
if dia_nombre == "Domingo":
    print("Hoy es domingo. La planta está cerrada. ¡Vaya a descansar!")

model = cp_model.CpModel()
HORIZONTE = 60*8

In [27]:
# 1. FILTRAR OPERADORES DISPONIBLES
# Solo los que no descansan hoy y están activos
operadores_disponibles = df_operadores[(df_operadores['DIA_DESCANSO'] != dia_nombre) & 
                            (df_operadores['ESTATUS'] == 'ACTIVO')].to_dict('records')

display(df_operadores[df_operadores['DIA_DESCANSO'] != dia_nombre]['AREA'].value_counts())
display(df_operadores.head(5))

AREA
ESTRUCTURA Y DISEÑO    6
DETALLADO              4
LAMINADO               3
ARMADO Y TAPIZADO      3
PINTURA                2
ELÉCTRICO              2
FIBRAS                 2
GENERAL                1
Name: count, dtype: int64

,ID_OPERADOR,OPERADOR,AREA,ID_DIA_DESCANSO,DIA_DESCANSO,FECHA_ACTUALIZACION_DESCANSO,FECHA_INGRESO,ESTATUS
0,1,ABIZAID ARAGÓN,ARMADO Y TAPIZADO,1,Lunes,2026-04-19,1900-01-01,ACTIVO
1,2,ANAYELI LÓPEZ,GENERAL,6,Sábado,2026-04-19,1900-01-01,ACTIVO
2,3,AQUILEO GUTIÉRREZ,ESTRUCTURA Y DISEÑO,1,Lunes,2026-04-19,1900-01-01,ACTIVO
3,4,DAVID MOLINA,LAMINADO,2,Martes,2026-04-19,1900-01-01,ACTIVO
4,5,DAVID SALDÍVAR,PINTURA,2,Martes,2026-04-19,1900-01-01,ACTIVO


In [ ]:
# 2. PROCESAMIENTO DE TAREAS Y UNIDADES
task_vars = {}

# Función para convertir "1_10" en (1, 10) para comparar correctamente
def key_to_tuple(k):
    try:
        if pd.isna(k) or str(k).strip() == "" or k == "-":
            return (0, 0)
        return tuple(map(int, str(k).split('.')))
    except:
        return (0, 0)

for _, bus in df_wip.iterrows():
    u_id = bus['ID_UNIDAD']
    ultimo_id = str(bus['KEY_SUBPROCESO_MAS_RECIENTE']).strip()
    
    # 1. Calculamos prioridad
    f_ingreso = datetime.strptime(bus['FECHA_INGRESO_PRODUCCION'], "%d/%m/%Y")
    dias_en_planta = (fecha_hoy - f_ingreso).days
    peso_prioridad = dias_en_planta * 100 

    # 2. Filtramos actividades (Comparación de tuplas)
    val_ultimo = key_to_tuple(ultimo_id)
    
    # Creamos una máscara booleana para filtrar
    mask = df_lista_actividades['KEY_SUBPROCESO'].apply(key_to_tuple) > val_ultimo
    proximas_actividades = df_lista_actividades[mask].copy()

    # Ordenamos por la tupla para que 1_10 vaya después de 1_9
    proximas_actividades['temp_sort'] = proximas_actividades['KEY_SUBPROCESO'].apply(key_to_tuple)
    proximas_actividades = proximas_actividades.sort_values('temp_sort').head(3)
    display(proximas_actividades)

    # DEBUG: Si display no trae nada, imprime por qué
    if proximas_actividades.empty:
        print(f"Atención: {u_id} no tiene tareas pendientes después de {ultimo_id}")
        continue

    # 3. Asignación de tareas
    for _, tarea in proximas_actividades.iterrows():
        a_key = tarea['KEY_SUBPROCESO']
        
        # Limpieza de tiempo (evita el ValueError de float '')
        t_raw = str(tarea['TIEMPO']).strip()
        duracion_min = int(float(t_raw) * 60) if t_raw not in ['', 'nan', 'None'] else 0
        
        if duracion_min == 0: continue 

        # Buscar operadores (Limpiamos espacios para asegurar el match)
        area_tarea = str(tarea['AREA']).strip()
        operadores_aptos = [
            o['ID_OPERADOR'] for o in operadores_disponibles 
            if str(o['AREA']).strip() == area_tarea
        ]
        
        if not operadores_aptos:
            print(f"Aviso: Sin operadores para {area_tarea} (Unidad {u_id})")
            continue

        # Variables de OR-Tools
        suffix = f"{u_id}_{a_key}".replace(".", "_")
        start = model.NewIntVar(0, HORIZONTE, f'start_{suffix}')
        end = model.NewIntVar(0, HORIZONTE, f'end_{suffix}')
        interval = model.NewIntervalVar(start, duracion_min, end, f'inter_{suffix}')
        op_var = model.NewIntVarFromDomain(cp_model.Domain.FromValues(operadores_aptos), f'op_{suffix}')
        
        task_vars[(u_id, a_key)] = {
            'start': start, 'end': end, 'interval': interval, 
            'op': op_var, 'peso': peso_prioridad, 'dur': duracion_min,
            'dep': tarea['DEPENDENCIA']
        }

,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
58,9,8,9.8,DETALLADO Y SALIDA,Limpieza de vidrios y aspirado final,DETALLADO,2.5,9.7,"(9, 8)"
59,9,9,9.9,DETALLADO Y SALIDA,Retoque de tapas y firmas de calidad,DETALLADO,1.0,9.7,"(9, 9)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
53,9,3,9.3,DETALLADO Y SALIDA,Instalar calcas exteriores y biseles,DETALLADO,1.5,8.8,"(9, 3)"
54,9,4,9.4,DETALLADO Y SALIDA,Sellar estribos y aplicar capa de goma,DETALLADO,2.0,9.1,"(9, 4)"
55,9,5,9.5,DETALLADO Y SALIDA,Instalación de vidrios y parabrisas,DETALLADO,3.0,9.4,"(9, 5)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
51,9,1,9.1,DETALLADO Y SALIDA,Instalar chapa pistón y pistones aire,DETALLADO,2.0,8.8,"(9, 1)"
52,9,2,9.2,DETALLADO Y SALIDA,Preparar e instalar espejos,DETALLADO,1.5,8.8,"(9, 2)"
53,9,3,9.3,DETALLADO Y SALIDA,Instalar calcas exteriores y biseles,DETALLADO,1.5,8.8,"(9, 3)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
44,8,2,8.2,ARMADO E INTERIORES,Adaptar y soldar ductos de puertas,ARMADO Y TAPIZADO,2.0,8.1,"(8, 2)"
45,8,3,8.3,ARMADO E INTERIORES,Instalar y pijar ductería general,ARMADO Y TAPIZADO,3.0,8.2,"(8, 3)"
46,8,4,8.4,ARMADO E INTERIORES,Instalar falleba y aro de falleba,ARMADO Y TAPIZADO,1.0,8.3,"(8, 4)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
37,7,2,7.2,PREPARACIÓN Y PINTURA,Aplicar fondo plaste en carrocería,PINTURA,3.0,7.1,"(7, 2)"
38,7,3,7.3,PREPARACIÓN Y PINTURA,Aplicar fondo epóxico en techo,PINTURA,1.5,7.1,"(7, 3)"
39,7,4,7.4,PREPARACIÓN Y PINTURA,Empapelar unidad completa,PINTURA,2.0,7.2,"(7, 4)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
24,4,8,4.8,LAMINADO Y CNC,Instalar refuerzo de tapa diesel y Adblue,LAMINADO,1.0,4.7,"(4, 8)"
25,4,9,4.9,LAMINADO Y CNC,Fabricar e instalar postes de cabina,LAMINADO,2.0,4.8,"(4, 9)"
27,5,1,5.1,SISTEMA ELÉCTRICO BASE,Fabricar arnés de techo,ELÉCTRICO,2.0,,"(5, 1)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
18,4,2,4.2,LAMINADO Y CNC,Fabricar charola para puerta de baterías,LAMINADO,0.5,,"(4, 2)"
19,4,3,4.3,LAMINADO Y CNC,Fabricar charola para filtro de aire,LAMINADO,0.5,,"(4, 3)"
20,4,4,4.4,LAMINADO Y CNC,Cortar y doblar charola de puertas acceso,LAMINADO,1.0,,"(4, 4)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
12,3,3,3.3,ENSAMBLE DE ESQUELETO,Calibrar conjunto de escuadras de nivel,ESTRUCTURA Y DISEÑO,1.0,3.2,"(3, 3)"
13,3,4,3.4,ENSAMBLE DE ESQUELETO,Habilitado de perfil 1 1/2 para filtro aire,ESTRUCTURA Y DISEÑO,0.5,3.3,"(3, 4)"
14,3,5,3.5,ENSAMBLE DE ESQUELETO,Nivelar y soldar tensores frontales,ESTRUCTURA Y DISEÑO,1.5,3.4,"(3, 5)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
4,1,5,1.5,PROCESAMIENTO DE FIBRA,Detallado de piezas de fibra (Lijado base),FIBRAS,2.0,1.4,"(1, 5)"
5,2,1,2.1,PREPARACIÓN ESTRUCTURAL,Lavado de chassis y remoción de parafinas,ESTRUCTURA Y DISEÑO,1.5,,"(2, 1)"
6,2,2,2.2,PREPARACIÓN ESTRUCTURAL,Colocar protección de radiador y mangueras,ESTRUCTURA Y DISEÑO,2.5,2.1,"(2, 2)"


,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,temp_sort
1,1,2,1.2,PROCESAMIENTO DE FIBRA,Aplicación de Gelcoat,FIBRAS,0.5,1.1,"(1, 2)"
2,1,3,1.3,PROCESAMIENTO DE FIBRA,Laminado de capas de fibra de vidrio,FIBRAS,2.0,1.2,"(1, 3)"
3,1,4,1.4,PROCESAMIENTO DE FIBRA,Desmolde y recorte de excedentes,FIBRAS,1.5,1.3,"(1, 4)"


In [41]:
# 3. RESTRICCIONES TÉCNICAS
    
# A. Flujo Serial: La tarea B solo empieza si la A terminó hoy
for (u_id, a_key), task in task_vars.items():
    dep_key = task['dep']
    if (u_id, dep_key) in task_vars:
        model.Add(task['start'] >= task_vars[(u_id, dep_key)]['end'])

# B. Exclusividad del Operador: No-Overlap (Cero duplicidad de tareas)
for op in operadores_disponibles:
    ops_intervals = []
    for (u_id, a_key), task in task_vars.items():
        # Variable booleana para verificar si el operador o es el elegido
        is_assigned = model.NewBoolVar(f'bool_{op["ID_OPERADOR"]}_{u_id}_{a_key}')
        model.Add(task['op'] == op['ID_OPERADOR']).OnlyEnforceIf(is_assigned)
        model.Add(task['op'] != op['ID_OPERADOR']).OnlyEnforceIf(is_assigned.Not())
            
        # Intervalo opcional
        opt_inter = model.NewOptionalIntervalVar(task['start'], task['dur'], task['end'], 
                                                    is_assigned, f'opt_{op["ID_OPERADOR"]}_{u_id}_{a_key}')
        ops_intervals.append(opt_inter)
        
    if ops_intervals:
        model.AddNoOverlap(ops_intervals)

In [42]:
# 4. OPTIMIZACIÓN: Maximizar avance en unidades críticas
if not task_vars:
    print("ERROR: No se generaron tareas. Verifique que las unidades tengan actividades pendientes en la secuencia.")
else:
    print(f"Intentando programar {len(task_vars)} tareas con {len(operadores_disponibles)} operadores.")

# Penalizamos el tiempo de finalización (queremos que terminen lo antes posible)
# Multiplicado por el peso de prioridad (días en planta)
model.Minimize(sum(task['end'] * task['peso'] for task in task_vars.values()))

# 5. RESULTADOS
solver = cp_model.CpSolver()
status = solver.Solve(model)

Intentando programar 29 tareas con 23 operadores.


In [43]:
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    data_final = []
    for (u_id, a_key), task in task_vars.items():
        if solver.Value(task['end']) > 0:
            op_id = solver.Value(task['op'])
            nombre_op = next(o['OPERADOR'] for o in operadores_disponibles if o['ID_OPERADOR'] == op_id)
            data_final.append({
                'Operador': nombre_op, 'Unidad': u_id, 'Actividad': a_key,
                'Inicio (Min)': solver.Value(task['start']), 'Fin (Min)': solver.Value(task['end'])
            })
            df_asignaciones = pd.DataFrame(data_final).sort_values(['Operador', 'Inicio (Min)'])
    
    print("No se pudo generar un plan de trabajo. Verifique disponibilidad de áreas.")

No se pudo generar un plan de trabajo. Verifique disponibilidad de áreas.


In [15]:
if status == cp_model.INFEASIBLE:
    print("EL MODELO ES INFACTIBLE. Causas probables:")
    # Revisar si hay alguna tarea sin operadores aptos
    for (u, a), info in task_vars.items():
        # Esto te dirá qué tarea específica está causando el bloqueo
        pass 
    print("- Verifique que no haya tareas que duren más de 9 horas.")
    print("- Verifique que haya al menos un operador disponible para cada área activa hoy.")